# Domain 1 – Tyre Degradation: Exploratory Data Analysis

This notebook explores the raw bronze lap data to understand:
- Lap time distributions per compound
- Stint length distributions
- Track-by-track variation in pace and degradation


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120

from src.utils.paths import BRONZE_LAPS, DOMAIN1_SILVER
from src.features.domain1_degradation import load_bronze_laps

print('Imports OK')

## 1. Load Bronze Lap Data

In [ ]:
laps = load_bronze_laps()
print(f'Shape: {laps.shape}')
print(f'Columns: {laps.columns.tolist()}')
laps.head()

## 2. Basic Statistics

In [ ]:
print('=== Dataset Overview ===')
print(f'Total laps:   {len(laps):,}')
print(f'Drivers:      {laps["Driver"].nunique()}')
print(f'Events:       {laps["EventName"].nunique()}')
print(f'Years:        {sorted(laps["year"].unique())}')
print()
print('Compound distribution:')
print(laps['Compound'].value_counts())
print()
print('Lap time statistics (seconds):')
laps.groupby('Compound')['LapTime'].describe().round(3)

## 3. Lap Time Distribution per Compound

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

compounds_slick = ['SOFT', 'MEDIUM', 'HARD']
laps_slick = laps[laps['Compound'].isin(compounds_slick)]

# Box plot
sns.boxplot(
    data=laps_slick,
    x='Compound', y='LapTime',
    order=compounds_slick,
    palette={'SOFT': '#e74c3c', 'MEDIUM': '#f39c12', 'HARD': '#95a5a6'},
    ax=axes[0]
)
axes[0].set_title('Lap Time Distribution by Compound')
axes[0].set_ylabel('Lap Time (s)')
axes[0].set_xlabel('Tyre Compound')

# Violin plot
sns.violinplot(
    data=laps_slick,
    x='Compound', y='LapTime',
    order=compounds_slick,
    palette={'SOFT': '#e74c3c', 'MEDIUM': '#f39c12', 'HARD': '#95a5a6'},
    ax=axes[1]
)
axes[1].set_title('Lap Time Violin Plot by Compound')
axes[1].set_ylabel('Lap Time (s)')
axes[1].set_xlabel('Tyre Compound')

plt.tight_layout()
plt.show()

## 4. Stint Length Distribution

In [ ]:
if 'Stint' in laps.columns and 'TyreLife' in laps.columns:
    stint_lengths = (
        laps_slick.groupby(['year', 'round_number', 'Driver', 'Stint'])['TyreLife']
        .max()
        .reset_index(name='stint_length')
    )

    fig, ax = plt.subplots(figsize=(12, 5))
    sns.histplot(data=stint_lengths, x='stint_length', bins=30, kde=True, ax=ax)
    ax.axvline(stint_lengths['stint_length'].median(), color='red',
               linestyle='--', label=f'Median = {stint_lengths["stint_length"].median():.1f} laps')
    ax.set_title('Stint Length Distribution (all slick compounds)')
    ax.set_xlabel('Stint Length (laps)')
    ax.set_ylabel('Count')
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(stint_lengths['stint_length'].describe().round(1))
else:
    print('Stint/TyreLife columns not available in current data.')

## 5. Track-by-Track Variation

In [ ]:
if 'EventName' in laps.columns:
    # Median lap time per track (normalised to remove absolute pace differences)
    track_stats = (
        laps_slick.groupby('EventName')['LapTime']
        .agg(['median', 'std'])
        .rename(columns={'median': 'median_lap', 'std': 'lap_std'})
        .reset_index()
        .sort_values('lap_std', ascending=False)
    )

    fig, ax = plt.subplots(figsize=(14, 7))
    sns.barplot(data=track_stats, x='EventName', y='lap_std', ax=ax, palette='Blues_r')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    ax.set_title('Lap Time Standard Deviation by Track (higher = more variable pace)')
    ax.set_xlabel('Event')
    ax.set_ylabel('Std Dev of Lap Time (s)')
    plt.tight_layout()
    plt.show()

    print('Top 5 most variable tracks:')
    print(track_stats.head())
else:
    print('EventName column not available.')

## 6. Lap Time Evolution over Tyre Life

In [ ]:
if 'TyreLife' in laps.columns:
    tyre_life_avg = (
        laps_slick[laps_slick['TyreLife'] <= 40]
        .groupby(['Compound', 'TyreLife'])['LapTime']
        .median()
        .reset_index()
    )

    fig, ax = plt.subplots(figsize=(14, 6))
    compound_colors = {'SOFT': '#e74c3c', 'MEDIUM': '#f39c12', 'HARD': '#95a5a6'}
    for compound, group in tyre_life_avg.groupby('Compound'):
        # Normalise to lap-1 pace for each compound
        base = group.iloc[0]['LapTime']
        ax.plot(
            group['TyreLife'],
            group['LapTime'] - base,
            label=compound,
            color=compound_colors.get(compound, 'grey'),
            linewidth=2,
            marker='o', markersize=3
        )
    ax.set_title('Median Lap Time Delta vs Lap 1 by Tyre Compound')
    ax.set_xlabel('Tyre Age (laps)')
    ax.set_ylabel('Delta vs Lap 1 (s)')
    ax.legend(title='Compound')
    ax.axhline(0, color='black', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('TyreLife column not available.')